# Qwen 3.5 9B | Arabic Agentic Coding Assistant (Cloud Training)

This notebook is specifically engineered for **Cloud Deployment**. It incorporates crucial dataset sanitation, unified XML Tool mapping, and all crash-resiliency lessons learned.

In [ ]:
# 1. Environment & Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Imports and Cloud Patches
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import unsloth
from unsloth import FastLanguageModel
import torch
import gc
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Fix FlashAttention availability globally (helpful if falling back to custom attention kernels)
import importlib
try:
    import fla
    import transformers.utils.import_utils as _iu
    _iu.is_flash_linear_attention_available = lambda: True
    import transformers.models.qwen3_5.modeling_qwen3_5 as _m
    importlib.reload(_m)
except ImportError:
    pass

In [ ]:
# 3. Model Initialization (Lesson: Native bf16, No 4-bit for Qwen)
print("=== Loading Qwen 3.5 9B Model ===")
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,      # LESSON: Disabled! Qwen3.5 suffers severe degradation under 4-bit
    load_in_16bit = True,      # LESSON: Native bf16
    fast_inference = False,
    device_map = "auto",
)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 4. Adapter Configuration
print("=== Adding LoRA Adapters ===")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 5. Dataset Loading & ChatML Validation Matrix
print("=== Loading Curated Arabic Agentic Dataset ===")
if not os.path.exists("qwen_unified_arabic_agent.jsonl"):
    raise FileNotFoundError("CRITICAL: 'qwen_unified_arabic_agent.jsonl' is missing from the cloud workspace! Upload it to continue.")

dataset = load_dataset('json', data_files='qwen_unified_arabic_agent.jsonl', split='train')

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant", "system": "system"}
)

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return tokenizer(text=texts, truncation=True, max_length=max_seq_length)

# Execute massive threaded dataset serialization
dataset = dataset.map(
    formatting_prompts_func, 
    batched=True, 
    remove_columns=dataset.column_names,
    num_proc=16
)

In [ ]:
# 6. SFT Training Execution
print("=== Configuring SFT Trainer ===")
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    max_seq_length = max_seq_length,
    dataset_num_proc = 16,
    packing = True, # Context stuffing acceleration
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        learning_rate = 2e-5,
        num_train_epochs = 1,
        logging_steps = 5,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_qwen35_9B_arabic_sft",
        save_strategy = "steps",
        save_steps = 200,
        report_to = "none",
        bf16 = True,
    ),
)

checkpoints = [d for d in os.listdir("outputs_qwen35_9B_arabic_sft") if d.startswith("checkpoint-")] if os.path.exists("outputs_qwen35_9B_arabic_sft") else []
if checkpoints:
    print(f"Resuming crash-resilient run from latest checkpoint...")
    trainer.train(resume_from_checkpoint=True)
else:
    trainer.train()

model.save_pretrained("qwen35_9b_arabic_lora")
tokenizer.save_pretrained("qwen35_9b_arabic_lora")